In [1]:
from pathlib import Path
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

In [ ]:
DATA_ROOT = Path("../Data")

PREDICTIONS_ROOT = Path("../Results/Experiment_1")


DATASETS = ["apples", "tomatoes"]

In [7]:
def show_coco_predictions(
    dataset_root: Path,
    pred_coco_json_path: Path,
    gt_coco_json_path: Path,
):
    """
    Visualize predictions and ground-truth for a COCO-style dataset.

    dataset_root: Path to the dataset directory (the parent of the image file_name paths)
    pred_coco_json_path: Path to the coco.json predictions file
    gt_coco_json_path: Path to the coco.json ground-truth file
    """

    def _norm_fname(fname: str) -> str:
        """
        Normalize COCO file_name so pred and gt can be matched.

        Examples:
            "images/test/xxx.png"        -> "images/test/xxx.png"
            "../images/test/xxx.png"     -> "images/test/xxx.png"
            "./images/test/xxx.png"      -> "images/test/xxx.png"
        """
        # Use Path to normalize, then strip leading ./ and ../
        p = Path(fname)
        # as_posix to keep forward slashes
        s = p.as_posix()
        # strip leading ./ or ../
        while s.startswith("./") or s.startswith("../"):
            if s.startswith("./"):
                s = s[2:]
            elif s.startswith("../"):
                s = s[3:]
        return s

    # ------------------------------------------------------------------
    # Load COCO jsons
    # ------------------------------------------------------------------
    with pred_coco_json_path.open() as f:
        pred_coco = json.load(f)

    with gt_coco_json_path.open() as f:
        gt_coco = json.load(f)

    # ------------------------------------------------------------------
    # Index images and categories
    # ------------------------------------------------------------------
    pred_images_by_id = {img["id"]: img for img in pred_coco["images"]}
    pred_cats_by_id = {c["id"]: c["name"] for c in pred_coco["categories"]}

    # Ground truth: index by normalized file_name so we can match to preds
    gt_images_by_file = {
        _norm_fname(img["file_name"]): img for img in gt_coco["images"]
    }
    gt_cats_by_id = {c["id"]: c["name"] for c in gt_coco["categories"]}

    # ------------------------------------------------------------------
    # Group annotations per image (pred & gt)
    # ------------------------------------------------------------------
    pred_ann_by_image = {}
    for ann in pred_coco["annotations"]:
        pred_ann_by_image.setdefault(ann["image_id"], []).append(ann)

    gt_ann_by_image = {}
    for ann in gt_coco["annotations"]:
        gt_ann_by_image.setdefault(ann["image_id"], []).append(ann)

    # ------------------------------------------------------------------
    # Visualize per prediction image
    # ------------------------------------------------------------------
    for image_id, pred_anns in pred_ann_by_image.items():
        img_info = pred_images_by_id[image_id]

        # Load image from prediction file_name (already correct for dataset_root)
        img_path = dataset_root / img_info["file_name"]
        img = Image.open(img_path).convert("RGB")
        draw = ImageDraw.Draw(img)

        # Find corresponding GT image by normalized file_name
        norm_fname = _norm_fname(img_info["file_name"])
        gt_img_info = gt_images_by_file.get(norm_fname, None)
        if gt_img_info is not None:
            gt_anns = gt_ann_by_image.get(gt_img_info["id"], [])
        else:
            gt_anns = []

        # ------------------------------------------------------------------
        # Draw ground-truth boxes (e.g., green)
        # ------------------------------------------------------------------
        for ann in gt_anns:
            x, y, w, h = ann["bbox"]
            label = gt_cats_by_id.get(ann["category_id"], str(ann["category_id"]))
            # GT typically has no score
            draw.rectangle([x, y, x + w, y + h], outline="green", width=2)
            text = f"GT: {label}"
            draw.text((x, max(0, y - 24)), text, fill="green")

        # ------------------------------------------------------------------
        # Draw prediction boxes (e.g., red)
        # ------------------------------------------------------------------
        for ann in pred_anns:
            x, y, w, h = ann["bbox"]
            label = pred_cats_by_id.get(ann["category_id"], str(ann["category_id"]))
            score = ann.get("score")

            draw.rectangle([x, y, x + w, y + h], outline="red", width=2)
            text = f"Pred: {label}" + (f" {score:.2f}" if score is not None else "")
            # Slightly offset text so GT and Pred texts don't fully overlap
            draw.text((x, max(0, y - 12)), text, fill="red")

        # ------------------------------------------------------------------
        # Show image
        # ------------------------------------------------------------------
        plt.figure(figsize=(10, 8))
        plt.title(img_info["file_name"])
        plt.imshow(img)
        plt.axis("off")
        plt.show()


In [ ]:
dataset = "apples"
dataset_root = DATA_ROOT / dataset 

coco_json_path = PREDICTIONS_ROOT / "apples_test_rex_omni_predictions.json"
gt_coco_json_path = DATA_ROOT / dataset / "annotations" / "instances_test.json"

show_coco_predictions(dataset_root, coco_json_path, gt_coco_json_path )